# Clusters checking and consolidation, and more filtering

In [ ]:

import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import networkx as nx

from Helper_Functions import *


## Load the initial data we produced 

In [ ]:

# Load saved Parquet files
# target_protein = None
# target_protein = "hba1c"
target_protein = "insulin"


wide_inxs=["person_id", "measurement_date", "visit_occurrence_id"]

if target_protein is None:
    file_long = "data\\aou_labs_long.parquet"
    file_wide = "data\\aou_labs_wide.parquet"
else:
    # file_long = f"data\\aou_{target_protein}_labs_long.parquet"
    # file_wide = f"data\\aou_{target_protein}_labs_wide.parquet"
    file_long = f"data\\aou_{target_protein}_labs_long_cleaned.parquet"
    file_wide = f"data\\aou_{target_protein}_labs_wide_cleaned.parquet"

print("Loading saved Parquet files...")
# df_long = pd.read_parquet(file_long, engine='pyarrow')
df_wide = pd.read_parquet(file_wide, engine='pyarrow')

# print(f"\n Loaded df_long: {df_long.shape[0]:,} rows x {df_long.shape[1]} columns")
print(f" Loaded df_wide: {df_wide.shape[0]:,} rows x {df_wide.shape[1]} columns")

df_wide.head()


In [ ]:

target_proteins = ['Glucose', 'HbA1c', 'Insulin', 'Adiponectin', 'Leptin', 'IGFBP1', 'IGFBP2', 'Resistin', 'Glucagon', 'GDF15']

present_targets = [col for col in target_proteins if col in df_wide.columns]

start_column=-1
for prot_name in present_targets:
    if start_column <= list(df_wide.columns[0:]).index(prot_name):
        start_column = list(df_wide.columns[0:]).index(prot_name) +1
    
print(f"start_column = {start_column}")



In [ ]:
present_targets
keep_columns = wide_inxs.copy()
keep_columns.extend(present_targets)
keep_columns

In [ ]:
# if target_protein is None:
#     start_column = 10
# if target_protein == "hba1c":
#     # start_column = 5
#     start_column = 8
# if target_protein == "insulin":
#     # start_column = 10
#     start_column = 8

In [ ]:

df_wide.iloc[:,:11]
df_wide.iloc[:5,9:20]


## Load the clusters data

In [ ]:
# ==========================================
# Example Usage Pipeline
# ==========================================

# 1. Load your actual lab measurements DataFrame (rows = patients, columns = lab names)
labs_df = df_wide

# 2. Load the clustered column mapping
if target_protein is None:
    clusters_df = pd.read_csv("data\\clustered_lab_columns_sapbert.csv")
else:
    clusters_df = pd.read_csv(f"data\\clustered_{target_protein}_lab_columns_sapbert.csv")
clusters_df

## Clusters sizes distribution

In [ ]:

# import seaborn as sns

# sns.kdeplot(data=clusters_df, x="sample_size", log_scale=True)
sns.histplot(data=clusters_df, x="sample_size", log_scale=True, kde=True)

clusters_df["sample_size"].describe(percentiles= [0.1,0.25,0.5,0.75,0.9])



### what clusters have sufficient size

In [ ]:

# drop all columns that have fewer than N samples

N = 30

good_clusters_df = clusters_df.loc[clusters_df["sample_size"] >= N]
good_clusters_df



In [ ]:

# # 2.1. add a column for the number of samples
# len(df_wide[clusters_df.iloc[0,0]].dropna())
# for inx in clusters_df.index:
#     clusters_df.loc[inx,"sample_size"] = len(df_wide[clusters_df.loc[inx,"column_name"]].dropna())

# clusters_df


## Inspect one sample cluster

In [ ]:
good_clusters_df["cluster_id"].value_counts()
target_cluster_id = int(good_clusters_df["cluster_id"].value_counts().index[0])
target_cluster_id

In [ ]:
# 3. Select a specific cluster to analyze (e.g., Cluster 0)
# target_cluster_id = 0
cluster_cols = good_clusters_df[good_clusters_df["cluster_id"] == target_cluster_id]["column_name"].tolist()

print(f"Analyzing Cluster {target_cluster_id} with columns: {cluster_cols}")

# 4. Compute EMD distance matrix
# Set normalize=True to compare shapes, normalize=False to compare absolute values
emd_results = compute_cluster_emd_matrix(labs_df, cluster_cols, normalize=False)

print("\nPairwise Earth Mover's Distance Matrix:")
# print(emd_results)
emd_results

In [ ]:

labs_df[cluster_cols[:2]].count()



### try to validate and split the sample cluster

In [ ]:
# Select a candidate cluster columns list from your database
candidate_cols = cluster_cols

# Validate the cluster
result = validate_and_split_cluster(
    df=labs_df,
    cluster_cols=candidate_cols,
    p_overlap_thresh=0.01,  # 1% minimum overlap
    q_lower=0.10,           # 10th percentile
    q_upper=0.90,           # 90th percentile
    emd_q_thresh_ratio=0.15,   # Max 15% EMD/Spread ratio
    emd_s_thresh_ratio=1.5, # Max 1.5 EMD/Self EMD Noise
    jsd_threshold=0.2,      # Max 0.2 JSD 
    num_bins=50,            # 50 bins for JSD calculation
    use_rule="emd_and_jsd",  # Pass both EMD and JSD
    trim_outliers=True,     # Trim outliers before computing EMD and JSD
    quantile_trim_bounds=(0.05, 0.95)  # Trim to the 5th and 95th percentiles before computing EMD and JSD
)

print(f"Is True Cluster? {result['is_true_cluster']}")
print("\nSub-clusters resulting from validation:")
for idx, sub in enumerate(result['sub_clusters'], 1):
    print(f"  Sub-cluster {idx}: {sub}")

print("\nPairwise Breakdown:")
# print(result['pairwise_report'][['column_1', 'column_2', 'overlap_pct', 'is_close_linear', 'is_close_log', 'compatible']])
result['pairwise_report'][['column_1', 'column_2', 'overlap_pct', 'is_close_linear', 'is_close_log', 'compatible']]

In [ ]:
result['pairwise_report']

In [ ]:

# i = -2
# sample_column = labs_df.loc[:,cluster_cols[i]].dropna()
# # print(f"number of samples = {len(sample_column)}")
# # print(f"samples mean = {sample_column.mean()}")
# # print(f"samples std = {sample_column.std()}")
# sample_column.describe()



## Run validation and graph splitting on all the semantic clusters

In [ ]:
new_cluster_records = []
next_new_cluster_id = 0

# Loop through each candidate cluster
for orig_cluster_id, group in good_clusters_df.groupby("cluster_id"):
    cols = group["column_name"].tolist()
    
    if len(cols) > 1:
        # Run validation and graph splitting
        validation_res = validate_and_split_cluster(
            df=labs_df,
            cluster_cols=cols,
            p_overlap_thresh=0.01,  # 1% minimum overlap
            q_lower=0.10,           # 10th percentile
            q_upper=0.90,           # 90th percentile
            emd_q_thresh_ratio=0.15,   # Max 15% EMD/Spread ratio
            emd_s_thresh_ratio=1.5, # Max 1.5 EMD/Self EMD Noise
            jsd_threshold=0.2,      # Max 0.2 JSD 
            num_bins=50,            # 50 bins for JSD calculation
            use_rule="emd_and_jsd",  # Pass both EMD and JSD
            trim_outliers=True,     # Trim outliers before computing EMD and JSD
            quantile_trim_bounds=(0.05, 0.95)  # Trim to the 5th and 95th percentiles before computing EMD and JSD
        )
        sub_clusters = validation_res["sub_clusters"]
        was_split = not validation_res["is_true_cluster"]
    else:
        # Single-column cluster: no validation needed
        sub_clusters = [cols]
        was_split = False
    
    # Assign new, sequential cluster IDs to each sub-cluster group
    for sub_group in sub_clusters:
        for col_name in sub_group:
            new_cluster_records.append({
                "column_name": col_name,
                "original_cluster_id": orig_cluster_id,
                "new_cluster_id": next_new_cluster_id,
                "was_split": was_split,
                "sub_cluster_size": len(sub_group)
            })
        next_new_cluster_id += 1

# Convert records into a final clean DataFrame
final_clusters_df = pd.DataFrame(new_cluster_records)

# Sort for easy viewing
final_clusters_df = final_clusters_df.sort_values(
    by=["new_cluster_id", "column_name"]
).reset_index(drop=True)

final_clusters_df


### validated and splitted clusters information

In [ ]:
final_clusters_df["new_cluster_id"].nunique()
print(f"number of unique clusters = {final_clusters_df["new_cluster_id"].nunique()}")

len(final_clusters_df["new_cluster_id"])
print(f"number of columns left = {len(final_clusters_df["new_cluster_id"])}")

In [ ]:
# Print high-level summary
orig_count = good_clusters_df["cluster_id"].nunique()
new_count = final_clusters_df["new_cluster_id"].nunique()
split_clusters_count = final_clusters_df[final_clusters_df["was_split"]]["original_cluster_id"].nunique()

print(f"Original Clusters:     {orig_count}")
print(f"Final Post-Split:      {new_count}")
print(f"Clusters that Split:   {split_clusters_count}")

# Inspect specific columns from clusters that were split
split_preview = final_clusters_df[final_clusters_df["was_split"]]
print("\nPreview of Split Clusters:")
print(split_preview.head(15))

In [ ]:
cluster_summaries = []

for cluster_id, group in final_clusters_df.groupby("new_cluster_id"):
    cols = group["column_name"].tolist()
    
    if len(cols) > 1:
        emd_mat = compute_cluster_emd_matrix(labs_df, cols, normalize=False)
        
        # Get upper triangle of distance matrix (excluding diagonal zeros)
        upper_tri_indices = np.triu_indices_from(emd_mat, k=1)
        distances = emd_mat.values[upper_tri_indices]
        
        avg_emd = np.nanmean(distances)
        min_emd = np.nanmin(distances)
        max_emd = np.nanmax(distances)
    else:
        avg_emd, min_emd, max_emd = 0.0, 0.0, 0.0

    cluster_summaries.append({
        "cluster_id": cluster_id,
        "column_count": len(cols),
        "mean_emd": avg_emd,
        "min_emd": min_emd,
        "max_emd": max_emd
    })

summary_df = pd.DataFrame(cluster_summaries)
print(summary_df.head(10))
print(summary_df.shape)

In [ ]:
summary_df.loc[summary_df["column_count"]>1]

In [ ]:

# clusters_df
final_clusters_df


In [ ]:

i = 2
final_clusters_df["new_cluster_id"] == i
final_clusters_df.loc[final_clusters_df["new_cluster_id"] == i,"column_name"]
# final_clusters_df


## Merge columns of clusters that survived

In [ ]:
# Merge df_wide columns based on validated clusters
df_consolidated = merge_clustered_columns(
    df_wide=df_wide,
    final_clusters_df=final_clusters_df,
    unclustered_cols=keep_columns,
    naming_strategy="cluster_id"  
)

print(f"Original shape: {df_wide.shape}")
print(f"Merged shape:   {df_consolidated.shape}")
df_consolidated

### look at the merged columns

In [ ]:
df_consolidated.columns
cluster_columns = []
for col in df_consolidated.columns:
    if "|" in col:
        print(col)
        cluster_columns.append(col)

df_consolidated.loc[:,cluster_columns]


In [ ]:
df_consolidated.loc[:,cluster_columns].describe()

## Drop low data columns by perplexity

Drop columns with a low number of effective values

In [ ]:

cols_perplexity = get_columns_perplexity(df_consolidated, n_bins = -1)
cols_perplexity


### see columns with low perplexity

In [ ]:

min_individual_perplexity = 5 # min number of effective values

cols_perplexity.loc[cols_perplexity < min_individual_perplexity]



In [ ]:

# no_data_columns = df_consolidated.loc[:,df_consolidated.std() == 0].columns
low_data_columns = cols_perplexity.loc[cols_perplexity < min_individual_perplexity].index
low_data_columns


df_consolidated_final = df_consolidated.copy().drop(columns=low_data_columns)
df_consolidated_final



## Save Merged and Validated clusters after more cleaning

In [ ]:
import os

if target_protein is None:
    file_cleaned = "data\\aou_labs_wide_cleaned.parquet"
else:
    file_cleaned = f"data\\aou_{target_protein}_labs_wide_cleaned_clustered.parquet"

print("Saving Cleaned DataFrame to Parquet format...")
df_consolidated_final.to_parquet(file_cleaned, index=False, engine='pyarrow')

# 3. Print file sizes to confirm
size_wide_cleaned_mb = os.path.getsize(file_cleaned) / (1024 * 1024)

print("\n Data successfully saved to your workspace persistent disk!")
print(f"📁 {file_cleaned}: {size_wide_cleaned_mb:.2f} MB")
